# IPN: tạo cache chuỗi biến độ dài `variable_v1`

Notebook giữ nguyên toàn bộ frame keypoint, tính vận tốc trên các frame thật liên tiếp và tạo cache mới tách biệt với `T64`. `max_frames=256` ở bước này chỉ tạo metadata `exceeds_max_frames`; notebook **không cắt, không padding và không chia window**.

Điều kiện hoàn tất: train/val/test lần lượt có 2.443 / 589 / 1.101 cache, `failed=0`, mọi cache qua schema validation và fingerprint khớp JSON nguồn.

In [ ]:
from pathlib import Path
import datetime as dt
import json
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_BRANCH = 'HuyenHoang'
REPO_ROOT = Path('/content/dynamic_hand_gesture_recognition')
DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
KEYPOINT_ROOT = DATA_ROOT / 'keypoints'
CACHE_ROOT = DATA_ROOT / 'keypoint_tensor_cache'
MAX_FRAMES = 256
EXPECTED = {'train': 2443, 'val': 589, 'test': 1101}

def run(argv, cwd=None):
    print('$', ' '.join(map(str, argv)))
    subprocess.run([str(value) for value in argv], cwd=cwd, check=True)


## 1. Đồng bộ code

Nếu repository đã tồn tại, cell dùng `git pull --ff-only`; nếu chưa có, cell clone mới. Hãy commit/push file `variable_keypoint_cache.py` trước khi chạy notebook trên Colab.

In [ ]:
if (REPO_ROOT / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
else:
    run(['git', 'clone', '--filter=blob:none', '--branch', REPO_BRANCH, REPO_URL, REPO_ROOT])

MODULE = REPO_ROOT / 'final/code/src_neurosymbolic/variable_keypoint_cache.py'
assert MODULE.is_file(), (
    f'Thiếu code bước 3: {MODULE}. '
    f'Kiểm tra branch {REPO_BRANCH} và commit aec2a02 hoặc mới hơn.'
)
sys.path.insert(0, str(REPO_ROOT / 'final/code'))
print('Repository:', REPO_ROOT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip())

## 2. Kiểm tra đầu vào Drive

Cell dừng ngay nếu thiếu manifest, thư mục keypoint hoặc số dòng split không đúng. Nó cũng yêu cầu đủ 4.133 JSON trước khi bắt đầu để tránh tạo một cache tree không hoàn chỉnh.

In [ ]:
from src_neurosymbolic.models.keypoint_dataset import read_split_file, resolve_keypoint_path

assert KEYPOINT_ROOT.is_dir(), f'Không tìm thấy {KEYPOINT_ROOT}'
manifests = {split: DATA_ROOT / f'{split}.txt' for split in EXPECTED}
expected_sources = {}
for split, path in manifests.items():
    assert path.is_file(), f'Thiếu manifest: {path}'
    rows = [line for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    assert len(rows) == EXPECTED[split], (split, len(rows), EXPECTED[split])
    samples = read_split_file(path)
    resolved = [resolve_keypoint_path(source, KEYPOINT_ROOT, split).resolve() for source, _ in samples]
    assert len(resolved) == len(set(resolved)) == EXPECTED[split], f'Duplicate source ở {split}'
    expected_sources[split] = set(resolved)

for split, expected_paths in expected_sources.items():
    actual_paths = {path.resolve() for path in (KEYPOINT_ROOT / split).rglob('*.json')}
    missing = expected_paths - actual_paths
    extra = actual_paths - expected_paths
    assert not missing, f'{split}: thiếu {len(missing)} JSON được manifest yêu cầu'
    print(split, 'manifest JSON verified:', len(expected_paths), '| extra JSON ignored:', len(extra))
    if extra:
        print('  extra examples:', [str(path) for path in sorted(extra)[:5]])
CACHE_ROOT.mkdir(parents=True, exist_ok=True)


## 3. Tạo cache `variable_v1`

Có thể chạy lại an toàn: cache chỉ được skip khi schema, fingerprint JSON và metadata threshold đều khớp. Cache stale sẽ được dựng lại. CLI trả lỗi nếu bất kỳ sample nào thất bại.

In [ ]:
variable_root = CACHE_ROOT / 'variable_v1'
variable_root.mkdir(parents=True, exist_ok=True)
report_path = variable_root / 'CACHE_BUILD_REPORT.json'
report_tmp = variable_root / 'CACHE_BUILD_REPORT.json.tmp'
report_tmp.write_text(json.dumps({
    'status': 'RUNNING',
    'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
}, indent=2) + '\n', encoding='utf-8')
report_tmp.replace(report_path)

command = [
    sys.executable, MODULE,
    '--keypoint-root', KEYPOINT_ROOT,
    '--output-root', CACHE_ROOT,
    '--max-frames', MAX_FRAMES,
]
for split, manifest in manifests.items():
    command += ['--manifest', f'{split}={manifest}']
run(command, cwd=REPO_ROOT)

## 4. Audit toàn bộ cache và lưu report

Cell load từng cache bằng `weights_only=True`, kiểm tra schema, fingerprint, số frame và thống kê số clip vượt 256. Report JSON được lưu cạnh cache trên Drive.

In [ ]:
from src_neurosymbolic.variable_keypoint_cache import cache_matches_source, load_variable_cache

audit = {}
for split, expected in EXPECTED.items():
    cache_split = variable_root / split
    actual_cache_paths = {path.resolve() for path in cache_split.rglob('*.pt')}
    expected_cache_paths = {
        (variable_root / source_path.relative_to(KEYPOINT_ROOT.resolve())).with_suffix('.pt').resolve()
        for source_path in expected_sources[split]
    }
    missing = expected_cache_paths - actual_cache_paths
    extra = actual_cache_paths - expected_cache_paths
    assert not missing and not extra, f'{split}: cache missing={len(missing)}, extra={len(extra)}'
    cache_paths = sorted(actual_cache_paths)
    assert len(cache_paths) == expected
    lengths = []
    exceeds = 0
    for index, cache_path in enumerate(cache_paths, 1):
        payload = load_variable_cache(cache_path)
        source_path = KEYPOINT_ROOT / cache_path.relative_to(variable_root).with_suffix('.json')
        assert source_path.is_file(), source_path
        assert cache_matches_source(payload, source_path), f'Cache stale: {cache_path}'
        assert payload['exceeds_max_frames'] == (payload['length'] > MAX_FRAMES)
        lengths.append(payload['length'])
        exceeds += int(payload['exceeds_max_frames'])
        if index % 500 == 0:
            print(f'{split}: validated {index}/{expected}')
    audit[split] = {
        'clips': len(lengths),
        'min_frames': min(lengths),
        'max_frames': max(lengths),
        'exceeds_256': exceeds,
    }

report = {
    'status': 'PASS',
    'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'schema': 'variable_v1',
    'max_frames_metadata': MAX_FRAMES,
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip(),
    'splits': audit,
}
report_tmp.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
report_tmp.replace(report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Saved:', report_path)

## Kết quả hợp lệ

Notebook chỉ hoàn tất khi cell cuối in `status: PASS`. Không copy `variable_v1` đè lên `T64`; hai cache phục vụ hai pipeline đối chứng khác nhau. Bước 4 của kế hoạch mới kết nối cache này với Dataset/DataLoader và padding mask.